# A2-Lite 640×192 — recovery and selective A2 distillation pilot
Latest revision: **A2-LITE-640-RECOVERY-KD-2026-10-12-r1**

Student: the SAME speed-screened MobileNetV4 Medium / E1-D1 A2-Lite, 640×192, 256 channels, 50 deployed queries. Teacher: frozen original A2 epoch130, running its validated full-resolution ONNX graph on CPU. No MonoDGP, architecture sweep, Core ML, old virtual environment restore or CUDA extension compilation. Choose a fresh **GPU Colab runtime** for student training; A100 is preferred for turnaround. Torch/CUDA are not reinstalled.

The untrained Lite model's moderate AP was zero. First verify initial model parity and gradients, then attempt 128 updates on 16 TRAIN frames. Only if learning improves do we run a bounded paired pilot: 512 TRAIN frames (256 Pedestrian-containing + 256 Vehicle-only), five fixed epochs for GT-only control and GT + A2 guidance. Balanced frame groups do NOT imply equal object counts; encoded class counts are reported. Both arms start from the identical pruned initialization, NOT the tiny-fit checkpoint.

The model is kept in eval mode while autograd updates its weights: this selects the same batch-1/50-query path we deploy and freezes BatchNorm/dropout. GT uses device-safe native-style focal, box/center/GIoU, depth, dimension and angle instance losses. The optional dense depth-map auxiliary loss and 11 training query groups are not part of this first pilot. These are training-recipe choices, not new inference modules.

Teacher and student queries are matched through TRAIN ground-truth objects; query numbers are never assumed to align. Teacher class confidence ≥0.1 and 2D IoU≥0.5 are required. Each KD component must be at least 10% closer to GT than the current student, with class-balanced means. KD weight ramps to 0.25 over 128 updates; temperature is fixed at 2, not swept. No validation targets select guidance, learning rates, epochs or checkpoints. Each fixed endpoint is evaluated once on all 3,769 Chen validation frames using existing product AP_R40 and nearby recall.

Run sections 1–8 in order. Checkpoints save every 128 updates; interruption can redo up to 127 updates. After a Colab reset rerun sections 1–5 before resuming section 6. Small-set failure or insufficient teacher coverage are findings: later training is skipped, and you still return the ZIP. A complete pilot is not full training or deployment qualification. Re-export and iPhone accuracy/speed/camera tests come after review.

## 1. Mount Drive, paths and all helpers
Use the original A2-Lite Drive folder: its initialization checkpoint was excluded from the download ZIP. Correct paths here if you relocated artifacts. No cells from older notebooks are needed.

In [ ]:
# A2-LITE-640-RECOVERY-KD-2026-10-12-r1
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, re, shlex, subprocess, sys, zipfile
REVISION='A2-LITE-640-RECOVERY-KD-2026-10-12-r1'
RUN_ID='a2_lite_640_recovery_kd_r1'
if not re.fullmatch(r'[a-z0-9_]+',RUN_ID): raise ValueError('Use lowercase letters, digits and underscores')
MOBILE_REPO=Path('/content/mobile_adas3d')
BASELINE=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite/a2_lite_e1_d1_cpu_screen_r1')
INITIALIZATION=BASELINE/'a2_lite_initialization.pth'
ACCURACY=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_accuracy/a2_lite_640_accuracy_r1')
COMPACT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_resolution/a2_lite_resolution_cpu_screen_r1')
TEACHER_EXPORT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_a2_m68_onnx_cpu/m68_a2_onnx_cpu_r3/export')
SPLITS=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
SOURCE_REPO=Path('/content/MonoDETR_A2LITE_RECOVERY_'+RUN_ID)
DATASET=Path('/content/kitti_a2_lite_accuracy_recovery_'+RUN_ID)
CANDIDATES=[Path(p) for p in ('/content/kitti_a2_lite_accuracy_a2_lite_640_accuracy_r1','/content/kitti_m68','/content/kitti_m67','/content/kitti','/content/monodetr_kitti_a2','/content/drive/MyDrive/datasets/kitti')]
OUTPUT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_recovery')/RUN_ID
PYTHON=Path(sys.executable);SCRIPT=MOBILE_REPO/'scripts/recover_a2_lite_640.py'
LOGS=OUTPUT/'colab_logs';LOGS.mkdir(parents=True,exist_ok=True)
READY=False;TINY_PASSED=False;KD_ALLOWED=False
def run_logged(command,name,cwd=None):
    command=list(map(str,command));path=LOGS/f"{name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.log"
    print('+',shlex.join(command),'\nDurable log:',path,flush=True);tail=deque(maxlen=40)
    with path.open('w') as log:
        log.write(shlex.join(command)+'\n');log.flush()
        process=subprocess.Popen(command,cwd=str(cwd) if cwd else None,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line.rstrip())
        except BaseException:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
            raise
        finally:process.stdout.close()
        code=process.wait();log.write(f'\nEXIT CODE: {code}\n');log.flush()
    if code:raise RuntimeError(f'Exit {code}; full log={path}\n'+'\n'.join(tail))
    return path
def stage(name,*extra):
    if not READY and name!='bundle':raise RuntimeError('Run sections 1 and 2 first')
    return run_logged([PYTHON,'-X','faulthandler','-u',SCRIPT,name,'--output',OUTPUT,'--device','cuda',*extra],name,MOBILE_REPO)
print(REVISION,'\nOutput:',OUTPUT,'\nExact 640×192 student; original A2 teacher; no phone connection needed.')


## 2. Update main, dependencies and GPU check
No old virtual environment, Torch/CUDA replacement, nvcc or native extension build. After a reset, saved CPU package versions are restored from THIS recovery manifest; GPU allocation is not mistaken for model identity.

In [ ]:
READY=False
if not MOBILE_REPO.exists():run_logged(['git','clone','--branch','main','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone')
dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE_REPO,text=True).strip()
if dirty:raise RuntimeError('Preserve tracked edits and use a clean main checkout.\n'+dirty)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch!='main':raise RuntimeError(f'Use main, not {branch!r}')
run_logged(['git','pull','--ff-only','origin','main'],'update',MOBILE_REPO)
if not SCRIPT.is_file():raise FileNotFoundError(SCRIPT)
for path in (BASELINE,INITIALIZATION,ACCURACY,COMPACT,TEACHER_EXPORT,SPLITS/'train.txt',SPLITS/'val.txt'):
    if not path.exists():raise FileNotFoundError(f'{path}: correct the path in section 1')
versions={'numpy':'2.1.3','timm':'1.0.27','onnxruntime':'1.30.0','pillow':'11.3.0','scipy':'1.16.3','opencv-python-headless':None}
manifest_path=OUTPUT/'recovery_manifest.json'
if manifest_path.exists():
    saved=json.loads(manifest_path.read_text())['cpu_environment']
    for name in versions:versions[name]=saved[name]
packages=[name+('=='+version if version else '') for name,version in versions.items()]+['pyyaml','scikit-image']
run_logged([PYTHON,'-m','pip','install','-q',*packages],'dependencies')
run_logged([PYTHON,'-c',"import torch,timm,onnxruntime as ort;assert torch.cuda.is_available(),'Select a fresh GPU Colab runtime';torch.cuda.init();assert timm.__version__=='1.0.27';assert ort.__version__=='1.30.0';print(torch.__version__,torch.cuda.get_device_name(0),'portable tensor training, no native extension')"],'runtime_probe')
READY=True


## 3. Tests, frozen inputs and private data staging
Restore only a dedicated source checkout at the reviewed MonoDETR commit. Validate exact checkpoint/export/report hashes. Copy/hash all train+validation files locally to reduce Drive I/O. Select the 512-image pilot only from training labels. Previous experiments are never rewritten.

In [ ]:
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_recovery.py','-v'],'tests',MOBILE_REPO)
extra=[]
for candidate in CANDIDATES:extra += ['--candidate',candidate]
stage('prepare','--baseline',BASELINE,'--initialization',INITIALIZATION,'--accuracy',ACCURACY,'--compact',COMPACT,'--teacher-export',TEACHER_EXPORT,'--repo',SOURCE_REPO,'--dataset',DATASET,'--split-dir',SPLITS,*extra)


## 4. Initial parity, real gradient smoke and tiny-set learning
Verify all 16 original 640 fixtures before optimization, then two GPU GT steps on a disposable copy. Run 128 GT-only updates on 16 train frames from the original initialization. Gate: GT objective drops at least 30%, normalized geometry error drops at least 10% overall AND for each class, and both classes have ≥8 encoded GT objects. This is an overfit diagnostic, NOT validation accuracy. A failed gate is saved as a result and skips the paired pilot.

In [ ]:
stage('smoke')
stage('tiny')
TINY=json.loads((OUTPUT/'tiny_fit.json').read_text());TINY_PASSED=bool(TINY['passed'])
print(json.dumps({k:TINY[k] for k in ('passed','initial','final','gt_loss_ratio','geometry_error_ratio','geometry_ratio_per_class','evaluation_scope')},indent=2))
if not TINY_PASSED:print('STOP before paired training. Run section 8 and return the ZIP; do not lower the gate or overwrite old results.')


## 5. Frozen A2 guidance — TRAIN images only
Cache the original full-resolution A2 once for the 512 selected train frames; this runs on CPU and may be slow. Cache records are resumable and bind all source bytes and native GT encoding. Count class-correct teacher/GT matches with score≥0.1 and 2D IoU≥0.5. Require ≥16 qualified matches per class for the KD arm. Geometry guidance is further gated dynamically during training.

In [ ]:
KD_ALLOWED=False
if TINY_PASSED:
    stage('cache')
    CACHE=json.loads((OUTPUT/'teacher_cache_complete.json').read_text());KD_ALLOWED=bool(CACHE['kd_pilot_authorized'])
    print(json.dumps({k:CACHE[k] for k in ('encoded_gt_counts','qualified_teacher_matches','kd_pilot_authorized')},indent=2))
    if not KD_ALLOWED:print('Teacher coverage is insufficient; only the GT-only control may proceed. Return the evidence for review.')
else:print('Skipped: tiny learnability gate did not pass.')


## 6. Matched five-epoch pilot — resumable checkpoints
Two arms, identical frozen starting weights, frame order, learning rates and update budget. The tiny-fit weights are NOT reused. Each arm gets 2,560 updates. No augmentation or hyperparameter sweep in this first pilot. Instance GT remains primary; KD uses object-aligned, class-balanced, current-student-lag masks. If interrupted, rerun unchanged; after reset rerun sections 1–5 first.

In [ ]:
if TINY_PASSED:
    stage('train','--role','control')
    if KD_ALLOWED:stage('train','--role','kd')
else:print('No paired training started: return the tiny-fit results.')


## 7. Full Chen validation of fixed endpoints
All 3,769 validation frames, the same native-style text decoder, product 3D/BEV AP_R40 and nearby recall as the reviewed audit. Validation does not choose checkpoints or training settings. Endpoint inference is resumable. Failed accuracy targets are findings, not execution errors. Export parity and iPhone camera/thermal tests are still required after any accuracy improvement.

In [ ]:
if TINY_PASSED:
    stage('evaluate','--role','control')
    if KD_ALLOWED:
        stage('evaluate','--role','kd')
        stage('compare')
        REPORT=json.loads((OUTPUT/'recovery_comparison.json').read_text())
        print(json.dumps({k:REPORT[k] for k in ('kd_minus_control','control_minus_untrained','interpretation')},indent=2))
    else:print('GT-only endpoint evaluated; no KD comparison is claimed.')
else:print('Validation skipped because the training learnability gate failed.')


## 8. Download results — also after a failure
Return a2_lite_recovery_results.zip. It contains receipts, metrics, predictions, cache identities and combined logs, not large checkpoints, ONNX assets or images. Checkpoints stay on Drive for resume and later export. No iPhone deployment or full-length training is authorized by this pilot.

In [ ]:
BUNDLE=OUTPUT/'a2_lite_recovery_results.zip'
try:
    if not SCRIPT.is_file():raise RuntimeError('Script unavailable; package logs with the standard library')
    stage('bundle')
except Exception as error:
    print('Bundle helper unavailable:',error,'— packaging readable results/logs directly.')
    with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and 'assets' not in path.relative_to(OUTPUT).parts and path.suffix in ('.json','.csv','.txt','.log'):archive.write(path,path.relative_to(OUTPUT))
print(BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
